# WikiNews NLP Project — Text Preprocessing and Named Entity Recognition

This notebook applies linguistic preprocessing and Named Entity Recognition (NER) to the 60-article analysis sample.

The main goals are to:

- segment news text into sentences and tokens,
- identify grammatical roles using part-of-speech tagging,
- inspect lemmas and dependency relations,
- extract named entities using spaCy,
- associate each entity with its source article and metadata,
- prepare structured entity data for later analysis and error investigation.

## 1. Imports and Configuration

In [2]:
from pathlib import Path
import sys

import pandas as pd
import spacy

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

In [3]:
nlp = spacy.load("en_core_web_sm")

print("spaCy version:", spacy.__version__)
print("Pipeline:", nlp.pipe_names)

spaCy version: 3.8.16
Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## 2. Load the Analysis Sample

The detailed NLP analysis uses the balanced sample of 60 English WikiNews articles, with 15 articles from each selected news category.

In [4]:
DATA_PATH = PROJECT_ROOT / "data/processed/analysis_sample.csv"

analysis_df = pd.read_csv(DATA_PATH)

print("Dataset shape:", analysis_df.shape)
print("\nCategory distribution:")
print(analysis_df["primary_category"].value_counts())

Dataset shape: (60, 9)

Category distribution:
primary_category
Politics and conflicts    15
Economy and business      15
Science and technology    15
Sports                    15
Name: count, dtype: int64


In [5]:
sample_article = analysis_df.iloc[0]

print("Title:", sample_article["title"])
print("Category:", sample_article["primary_category"])
print("Date:", sample_article["date"])
print("\nText preview:\n")
print(sample_article["text"][:1000])

Title: Brazilian President party received money from FARC, say documents
Category: Politics and conflicts
Date: 2005-03-15

Text preview:

Brazil —
Documents of the Brazilian Agency of intelligence (Abin) say that the Workers' Party received 5 million dollars to be used by political campaign of  candidates in 2002 from the Colombian communist armed group Revolutionary Armed Forces of Colombia (FARC-EP). The information was reported by the Brazilian magazine Veja that circulates this week in a headline story called "FARC's tentacles in Brazil". According to the magazine, reporters had access to documents of Abin that described the liaisons  between the Workers' Party (PT) and the Colombian guerrilla movement FARC. The Workers' Party (PT) is the party of the Brazilian President Luiz Inácio Lula da Silva.  PT is one of the biggest left-wing parties in Latin America and at this moment the strongest and more organized party from Brazil. It is one of the political Brazilian parties that has 

## 3. Linguistic Preprocessing

spaCy is used to identify:

- sentences,
- tokens,
- lemmas,
- part-of-speech tags,
- dependency relations,
- stopwords,
- punctuation.

In [6]:
doc = nlp(sample_article["text"])

print("Number of sentences:", len(list(doc.sents)))
print("Number of tokens:", len(doc))

Number of sentences: 137
Number of tokens: 2889


In [7]:
token_rows = []

for token in list(doc)[:40]:
    token_rows.append(
        {
            "token": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "dependency": token.dep_,
            "is_stop": token.is_stop,
            "is_punctuation": token.is_punct,
        }
    )

token_df = pd.DataFrame(token_rows)

token_df

,token,lemma,pos,dependency,is_stop,is_punctuation
0,Brazil,Brazil,PROPN,nsubj,False,False
1,—,—,PUNCT,punct,False,True
2,\n,\n,SPACE,dep,False,False
3,Documents,document,NOUN,appos,False,False
4,of,of,ADP,prep,True,False
5,the,the,DET,det,True,False
6,Brazilian,Brazilian,PROPN,compound,False,False
7,Agency,Agency,PROPN,pobj,False,False
8,of,of,ADP,prep,True,False
9,intelligence,intelligence,NOUN,pobj,False,False


In [8]:
sentences = [sentence.text for sentence in doc.sents]

for index, sentence in enumerate(sentences[:5], start=1):
    print(f"Sentence {index}: {sentence}\n")

Sentence 1: Brazil —
Documents of the Brazilian Agency of intelligence (Abin) say that the Workers' Party received 5 million dollars to be used by political campaign of  candidates in 2002 from the Colombian communist armed group Revolutionary Armed Forces of Colombia (FARC-EP).

Sentence 2: The information was reported by the Brazilian magazine Veja that circulates this week in a headline story called "FARC's tentacles in Brazil".

Sentence 3: According to the magazine, reporters had access to documents of Abin that described the liaisons  between the Workers' Party (PT) and the Colombian guerrilla movement FARC.

Sentence 4: The Workers' Party (PT) is the party of the Brazilian President Luiz Inácio Lula da Silva.  

Sentence 5: PT is one of the biggest left-wing parties in Latin America and at this moment the strongest and more organized party from Brazil.



## 4. Named Entity Recognition

spaCy's NER model is used to identify entities such as people, organisations, locations, events, dates, and monetary values.

In [9]:
entity_rows = []

for entity in doc.ents:
    entity_rows.append(
        {
            "entity": entity.text,
            "entity_type": entity.label_,
        }
    )

sample_entities = pd.DataFrame(entity_rows)

sample_entities.head(20)

,entity,entity_type
0,Brazil,GPE
1,the Brazilian Agency,ORG
2,Abin,PERSON
3,Party,ORG
4,5 million dollars,MONEY
5,2002,DATE
6,Colombian,NORP
7,Revolutionary Armed Forces of Colombia,ORG
8,Brazilian,NORP
9,Veja,PERSON


In [10]:
sample_entities["entity_type"].value_counts()

entity_type
ORG            100
PERSON          65
GPE             45
CARDINAL        41
NORP            39
DATE            29
LOC              5
MONEY            4
ORDINAL          4
WORK_OF_ART      4
QUANTITY         2
FAC              2
TIME             1
Name: count, dtype: int64

## 5. Extract Named Entities Across All Articles

Named entities are extracted from all 60 selected articles. Each entity is linked back to its article metadata, including title, publication date, category, and URL.

In [11]:
all_entities = []

for _, row in analysis_df.iterrows():
    doc = nlp(row["text"])

    for entity in doc.ents:
        all_entities.append(
            {
                "pageid": row["pageid"],
                "title": row["title"],
                "date": row["date"],
                "primary_category": row["primary_category"],
                "url": row["url"],
                "entity": entity.text,
                "entity_type": entity.label_,
            }
        )

entities_df = pd.DataFrame(all_entities)

print("Total extracted entities:", len(entities_df))
print("Unique entities:", entities_df["entity"].nunique())

entities_df.head(20)

Total extracted entities: 2727
Unique entities: 1550


,pageid,title,date,primary_category,url,entity,entity_type
0,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Brazil,GPE
1,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,the Brazilian Agency,ORG
2,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Abin,PERSON
3,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Party,ORG
4,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,5 million dollars,MONEY
5,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,2002,DATE
6,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Colombian,NORP
7,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Revolutionary Armed Forces of Colombia,ORG
8,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Brazilian,NORP
9,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Veja,PERSON


In [12]:
entities_df["entity_type"].value_counts()

entity_type
ORG            731
PERSON         439
DATE           376
GPE            357
CARDINAL       288
NORP           180
ORDINAL         87
LOC             54
TIME            44
PERCENT         38
MONEY           33
WORK_OF_ART     20
EVENT           20
FAC             19
QUANTITY        16
PRODUCT         16
LANGUAGE         7
LAW              2
Name: count, dtype: int64

In [13]:
entities_df.groupby(
    "primary_category"
)["entity"].count().sort_values(ascending=False)

primary_category
Politics and conflicts    908
Sports                    736
Economy and business      582
Science and technology    501
Name: entity, dtype: int64

## 6. Save Structured NER Results

The extracted entities are saved so that they can be reused in the next notebook for aggregated NER analysis, temporal analysis, and error investigation.

In [14]:
OUTPUT_PATH = PROJECT_ROOT / "data/processed/entities.csv"

entities_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("Saved:", OUTPUT_PATH)
print("Shape:", entities_df.shape)

Saved: /workspaces/rigoel-NLP.AI.2.5/data/processed/entities.csv
Shape: (2727, 7)


In [15]:
saved_entities = pd.read_csv(OUTPUT_PATH)

print(saved_entities.shape)
saved_entities.head()

(2727, 7)


,pageid,title,date,primary_category,url,entity,entity_type
0,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Brazil,GPE
1,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,the Brazilian Agency,ORG
2,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Abin,PERSON
3,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,Party,ORG
4,5796,Brazilian President party received money from ...,2005-03-15,Politics and conflicts,https://en.wikinews.org/wiki/Brazilian_Preside...,5 million dollars,MONEY


## Key Findings

- spaCy successfully identified grammatical structure through tokenization, lemmatization, part-of-speech tagging, and dependency parsing.
- Sentence segmentation was applied to long-form WikiNews articles.
- Named Entity Recognition identified people, organisations, geopolitical entities, locations, dates, events, and other entity types.
- Each extracted named entity is associated with its source article and corresponding metadata.
- The structured entity dataset is saved for aggregated analysis and NER error investigation in the next stage.